# Notebook 1 — Corpus & Preprocessing (Modules 1–2)
**Owner:** A1 (corpus + tokenization) and A2 (POS + n-grams).
**Time:** A1 ~ 45 min, A2 ~ 90 min.

**Reads:** `03_CORPUS/`, `04_GOLD/`, `02_SHARED_CODE/`
**Writes:** `10_PIPELINE_A/results_A/`

---
## Development history — read before rerunning

This notebook was iterated on 29–30.09.2026 while diagnosing corpus-fetching issues. The current version below is the corrected end-state. If someone reruns this from scratch they should get a clean corpus first time. This section documents what went wrong so the fixes are understandable and the same mistakes aren't reintroduced by well-meaning edits.

### Issue 1 — placeholder MIRACL qids fetched off-topic articles
The first version of §1's `QUERIES` table used placeholder qids (`train:1`, `train:2`, …) that were meant to be replaced with real video-game qids from `exp00_theme_queries.csv` before running §2. On the first run this step was missed, so §2 fetched Wikipedia articles by whatever qrels happened to be attached to those placeholder qids — mostly American football players and other unrelated topics.
**Fix.** §1 now programmatically discovers real video-game qids by scanning the MIRACL topics files with a keyword regex and auto-picks 12 balanced queries across clusters. No manual qid mapping needed.

### Issue 2 — Wikipedia API rate-limiting (HTTP 429)
§2's original fetcher hit the Wikipedia API once per article with only 0.1 s between requests. Wikipedia's default anonymous rate limit is around 200 requests/minute; sustained 10 req/sec earns HTTP 429 (Too Many Requests) responses and eventually an IP-level slowdown.
**Fix.** §2 now uses a `requests.Session()` for connection reuse, retries on 429 with the `Retry-After` header's wait, and paces requests at ~2.5 per second (`time.sleep(0.4)`).

### Issue 3 — batch extracts truncated by MediaWiki's API
An early attempted fix was to fetch 20 articles per API call via `pageids=X|Y|Z…`. That worked for the HTTP round-trip, but MediaWiki's `TextExtracts` extension truncates extracts to a short summary when multiple pages are requested unless `exlimit=1` is set — and `exlimit=1` limits the response to *one* full extract per call, defeating batching. Batches returned mostly empty `extract` fields, so almost every article failed our length filter.
**Fix.** §2 reverted to one API call per article with `exlimit=1` and the pacing above.

### Issue 4 — off-topic articles slipping into judged pools
MIRACL's relevance judgments include hard-negative articles — passages a native speaker judged *not* relevant. Some negatives for video-game queries were about unrelated topics (a footballer, an actor). If pulled uncritically they would pollute our corpus.
**Fix.** §2 applies a positive video-game keyword filter to each article's first 3000 characters before accepting it. Articles that don't mention any gaming term are dropped and logged, so the exclusion is auditable.

### What this means for the pipeline
None of these issues touched `vgir.py`, the preregistration, or any downstream analysis. They were all in the corpus-construction step. Once §2 produces a clean 40+ document corpus, the rest of the notebook runs unchanged.

---

## §0 Setup — run this FIRST every session

In [ ]:
!pip install -q nltk spacy tiktoken scikit-learn huggingface_hub pandas requests
!python -m spacy download en_core_web_sm --quiet

In [ ]:
# Mount Drive, resolve paths, patch vgir to write into our team's Drive layout.
import os, sys
from google.colab import drive
drive.mount('/content/drive')

BASE    = '/content/drive/MyDrive/NLP Assignment 1'
CODE    = f'{BASE}/02_SHARED_CODE'
DATA    = f'{BASE}/03_CORPUS'
GOLD    = f'{BASE}/04_GOLD'
RESULTS = f'{BASE}/10_PIPELINE_A/results_A'

for d in (DATA, RESULTS): os.makedirs(d, exist_ok=True)
sys.path.insert(0, CODE)

# vgir.P() is the library's path resolver. We monkey-patch it so 'data' reads/writes
# hit 03_CORPUS/, 'results' hit 10_PIPELINE_A/results_A/, and domain_lexicon.json
# comes from 01_FROZEN/. This keeps vgir.py unchanged (rule 2 of the team README:
# only one owner edits vgir.py, and only when necessary).
os.environ['VGIR_BASE'] = BASE
import vgir
def P(*parts):
    if not parts: return BASE
    head = parts[0]
    if head == 'data':    return os.path.join(DATA, *parts[1:])
    if head == 'results': return os.path.join(RESULTS, *parts[1:])
    if head == 'domain_lexicon.json': return f'{BASE}/01_FROZEN/domain_lexicon.json'
    return os.path.join(BASE, *parts)
vgir.P = P
def write_csv(rows, name, folder='results'):
    import pandas as pd
    target = RESULTS if folder == 'results' else os.path.join(BASE, folder)
    os.makedirs(target, exist_ok=True)
    df = pd.DataFrame(rows); df.to_csv(os.path.join(target, name), index=False); return df
vgir.write_csv = write_csv

print('BASE   :', BASE, '  exists=', os.path.isdir(BASE))
print('vgir   :', vgir.__file__)
print('lexicon:', vgir.P('domain_lexicon.json'), '  exists=', os.path.isfile(vgir.P('domain_lexicon.json')))

The three lines above must all show real paths that exist. If BASE.exists is False, the folder is in 'Shared with me' rather than 'My Drive'. Add a shortcut from Drive web UI (right-click the folder → Organise → Add shortcut → My Drive) and rerun this cell.

## §1 Load MIRACL topics + qrels, discover video-game queries

Rather than hard-code MIRACL qids, we scan the topics files at runtime for video-game queries with ≥ 2 positive relevance judgments. This is reproducible — anyone running the notebook gets the same query pool without needing an external lookup file.

In [ ]:
# Download topics + qrels from the miracl/miracl Hugging Face dataset (~200 KB total).
from huggingface_hub import hf_hub_download
for split in ('train', 'dev'):
    for kind in ('topics', 'qrels'):
        f = hf_hub_download('miracl/miracl', repo_type='dataset',
                            filename=f'miracl-v1.0-en/{kind}/{kind}.miracl-v1.0-en-{split}.tsv')
        os.system(f'cp "{f}" "{DATA}/"')
print('MIRACL files in', DATA, ':', [f for f in os.listdir(DATA) if f.endswith('.tsv')])

In [ ]:
# Load qrels into memory: qid -> {docid: relevance}.
# MIRACL uses 4-tuples (qid, iteration, docid, relevance) per TREC qrels format.
from collections import defaultdict

def load_qrels(path):
    d = defaultdict(dict)
    with open(path) as f:
        for line in f:
            qid, _, docid, rel = line.split()
            d[qid][docid] = int(rel)
    return d

qrels = {}
for split in ('train', 'dev'):
    for k, v in load_qrels(f'{DATA}/qrels.miracl-v1.0-en-{split}.tsv').items():
        qrels[f'{split}:{k}'] = v
print(f'loaded qrels for {len(qrels)} queries across train + dev')

In [ ]:
# Discover video-game queries by scanning topic text with a keyword regex.
# The regex was tuned during EXP-00 (see DECISION_LOG.md, DEC-001).
import re

VG_RX = re.compile(
    r"video ?games?|game ?boy|nintendo|playstation|xbox|\bsega\b|genesis|atari|\bconsoles?\b|"
    r"pok[e\u00e9]mon|zelda|metroid|\bhalo\b|\bdoom\b|wolfenstein|counter-strike|warcraft|"
    r"final fantasy|metal gear|tomb raider|silent hill|resident evil|mortal kombat|"
    r"gears of war|superhot|kingdom hearts|assassin'?s? creed|dragon quest|"
    r"monkey island|rock band|guitar hero|last of us|grand theft auto|total war|"
    r"\bsnes\b|super nintendo|\brpg\b|\bfps\b|shooter|adventure game|"
    r"point and click|first-person|third-person|role-playing", re.I)

def load_topics(path):
    with open(path) as f:
        return [line.rstrip('\n').split('\t', 1) for line in f if '\t' in line]

candidates = []
for split in ('train', 'dev'):
    for qid, text in load_topics(f'{DATA}/topics.miracl-v1.0-en-{split}.tsv'):
        if not VG_RX.search(text): continue
        key = f'{split}:{qid.strip()}'
        j = qrels.get(key, {})
        pos = [d.split('#')[0] for d, r in j.items() if r > 0]
        if len(pos) >= 2:
            candidates.append({'qid': key, 'query': text.strip(), 'n_pos': len(pos)})

print(f'{len(candidates)} video-game queries with >= 2 positive judgments')

In [ ]:
# Auto-pick 12 queries balanced across clusters (consoles, franchises, genre, sales, firsts).
from collections import Counter

def cluster(q):
    ql = q.lower()
    if any(w in ql for w in ['game boy','gameboy','nintendo','sega','xbox','playstation',
                              'atari','console','snes','super nintendo','genesis']):
        return 'consoles'
    if any(w in ql for w in ['shooter','rpg','role-playing','adventure','strategy','fps',
                              'first-person','third-person','point and click']):
        return 'genre'
    if any(w in ql for w in ['sold','sales','revenue','copies','units','million','billion']):
        return 'sales'
    if 'first' in ql or 'earliest' in ql or 'oldest' in ql:
        return 'firsts'
    return 'franchises'

for c in candidates: c['cluster'] = cluster(c['query'])

picked, per_cluster = [], Counter()
CLUSTER_MAX = 4
for c in sorted(candidates, key=lambda x: (-x['n_pos'], x['qid'])):
    if per_cluster[c['cluster']] >= CLUSTER_MAX: continue
    picked.append(c); per_cluster[c['cluster']] += 1
    if len(picked) == 12: break
if len(picked) < 12:   # top up if cluster caps prevented reaching 12
    for c in sorted(candidates, key=lambda x: -x['n_pos']):
        if c not in picked:
            picked.append(c)
            if len(picked) == 12: break

# QUERIES table: (miracl_qid, our_qid, query_text_as_indexed, type, cluster).
# All 12 are stored as 'keyword' type here. Boolean/phrase variants for the
# assignment's required query types are added in Notebook 2 where they matter,
# without polluting the corpus-building step.
QUERIES = [(c['qid'], f'Q{i+1:02d}', c['query'], 'keyword', c['cluster'])
           for i, c in enumerate(picked)]

import pandas as pd
print(f'cluster distribution: {dict(per_cluster)}\n')
pd.DataFrame(QUERIES, columns=['miracl_qid','our_qid','query','type','cluster'])

## §2 Fetch Wikipedia articles

For each of the 12 queries we take up to 4 positive-judged articles and up to 2 negative-judged articles from the qrels, then fetch each article's plain text from the MediaWiki API.

See the development-history section at the top of the notebook for why the fetcher is one-per-request (`exlimit=1`), paced at ~2.5 req/s, and applies a positive keyword filter.

In [ ]:
# Build the list of articles we want: 4 positives + 2 negatives per query.
wanted = []
for miracl_qid, our_qid, _, _, _ in QUERIES:
    j = qrels.get(miracl_qid, {})
    pos = [d.split('#')[0] for d, r in j.items() if r > 0]
    neg = [d.split('#')[0] for d, r in j.items() if r == 0]
    for a in pos[:4]: wanted.append((a, miracl_qid, 'positive', our_qid))
    for a in neg[:2]: wanted.append((a, miracl_qid, 'negative', our_qid))

uniq_ids = list({a for a, _, _, _ in wanted})
print(f'requested {len(wanted)} article slots; {len(uniq_ids)} unique articles to fetch')

In [ ]:
# Fetch each article via the MediaWiki API. Rate-limited, retry-safe, positively filtered.
import requests, time

S = requests.Session()
S.headers.update({'User-Agent': 'nlp-assignment-1/1.0 (educational use)'})

VG_KEYWORDS = re.compile(
    r"\b(video ?game|game ?boy|nintendo|sega|xbox|playstation|atari|arcade|"
    r"console|handheld|controller|gameplay|gaming|gamer|first-person|third-person|"
    r"role-playing|platformer|shooter|rpg|mmo|dlc|snes|nes|dreamcast|"
    r"multiplayer|singleplayer|open[- ]world|"
    r"released for|available on|ported to|developed by|published by|game engine|"
    r"pok[e\u00e9]mon|zelda|mario|sonic|halo|doom|wolfenstein|metroid|final fantasy|"
    r"tomb raider|total war|kingdom hearts|guitar hero)\b", re.I)

def fetch_one(pageid):
    """Fetch one Wikipedia article by pageid. Returns (title, extract) or None.
    Retries up to 4 times on transient failures; honours Retry-After on 429."""
    for attempt in range(4):
        try:
            r = S.get('https://en.wikipedia.org/w/api.php', params={
                'action': 'query', 'prop': 'extracts', 'explaintext': 1,
                'exsectionformat': 'plain', 'pageids': pageid,
                'format': 'json', 'formatversion': 2, 'redirects': 1,
                'exlimit': 1}, timeout=30)
            if r.status_code == 429:
                wait = int(r.headers.get('Retry-After', 30))
                print(f'  429 rate limit, waiting {wait}s'); time.sleep(wait); continue
            r.raise_for_status()
            for p in r.json().get('query', {}).get('pages', []):
                if 'missing' in p or 'invalid' in p: return None
                return p.get('title', 'Unknown'), p.get('extract', '')
        except Exception:
            time.sleep(2 ** attempt)
    return None

seen, kept, dropped = {}, [], []
print(f'fetching {len(uniq_ids)} articles at ~2.5 req/s...\n')

for i, aid in enumerate(uniq_ids, 1):
    if i % 10 == 0: print(f'  {i}/{len(uniq_ids)}...')
    result = fetch_one(aid)
    if result is None:
        dropped.append((aid, 'missing')); continue
    title, extract = result
    if len(extract) < 500:
        dropped.append((title, f'too_short ({len(extract)} chars)')); continue
    if not VG_KEYWORDS.search(extract[:3000]):
        dropped.append((title, 'no_gaming_keywords')); continue
    for a, srcq, role, our_qid in wanted:
        if a == aid and aid not in seen:
            seen[aid] = (title, extract, srcq, role, our_qid)
            kept.append(title); break
    time.sleep(0.4)

print(f'\n=== kept: {len(kept)} / dropped: {len(dropped)} ===')
print('\nKept (first 15):')
for t in kept[:15]: print(f'  keep: {t}')
print('\nDropped (first 10):')
for t, r in dropped[:10]: print(f'  drop [{r}]: {t}')

In [ ]:
# Write documents to disk with sequential IDs (D01..DNN) and a traceability manifest.
import datetime

# Wipe any earlier documents so IDs stay dense and there's no leftover contamination.
for f in os.listdir(DATA):
    if f.startswith('D') and f.endswith('.txt'):
        os.remove(f'{DATA}/{f}')

rows = []
for i, (aid, (title, body, srcq, role, our_qid)) in enumerate(sorted(seen.items()), 1):
    did = f'D{i:02d}'
    with open(f'{DATA}/{did}.txt', 'w') as f:
        f.write(title + '\n' + body)
    rows.append({'doc_id': did, 'wikipedia_pageid': aid, 'title': title,
                 'source_qid': srcq, 'source_our_qid': our_qid, 'role': role,
                 'body_chars': len(body)})

vgir.write_csv(rows, 'manifest.csv', folder='03_CORPUS')
with open(f'{DATA}/LOCKED_ON.txt', 'w') as f:
    f.write(str(datetime.date.today()))

print(f'wrote {len(rows)} documents to {DATA}')
import pandas as pd; pd.DataFrame(rows).head()

**Checkpoint.** Confirm 40–55 rows above. If fewer than 40, change `pos[:4]/neg[:2]` in the `wanted` list construction (three cells up) to `pos[:6]/neg[:4]` and rerun from that cell.

## §3 (A1) Corpus statistics — Table A

In [ ]:
# Reload vgir in case §0's monkey-patches got clobbered by an earlier reload.
import importlib; importlib.reload(vgir); vgir.P = P; vgir.write_csv = write_csv

docs = vgir.load_corpus(DATA)
print(f'{len(docs)} documents loaded.')
vgir.ensure_nltk()
from nltk.tokenize import sent_tokenize

stats = []
for did, d in docs.items():
    text = vgir.full_text(d); toks = vgir.tok_hybrid(text)
    stats.append({'doc_id': did, 'title': d['title'], 'chars': len(text),
                  'sentences': len(sent_tokenize(text)),
                  'tokens': len(toks), 'unique_tokens': len(set(toks))})
df = vgir.write_csv(stats, 'document_statistics.csv')
print('Totals:', df[['chars','sentences','tokens','unique_tokens']].sum().to_dict())
print('Corpus vocabulary:', len({t for d in docs.values() for t in vgir.tok_hybrid(vgir.full_text(d))}))
df.head()

## §4 (A1) Tokenization comparison — Ex. 3, Table B
Compares NLTK, spaCy, Custom (regex), and Hybrid tokenizers on the full corpus and on ≥ 5 domain-specific problem strings.

In [ ]:
import time

# Whole-corpus counts + timing per tokenizer.
rows = []
for name, fn in vgir.TOKENIZERS.items():
    if name == 'whitespace': continue
    all_toks = []
    t0 = time.perf_counter()
    for d in docs.values(): all_toks += fn(vgir.full_text(d))
    rows.append({'tokenizer': name, 'total_tokens': len(all_toks),
                 'unique_tokens': len(set(all_toks)),
                 'ms': round((time.perf_counter() - t0) * 1000, 1)})
vgir.write_csv(rows, 'tokenization_comparison.csv')

# Per-problem behaviour on hand-picked domain strings.
problems = ['Counter-Strike: Global Offensive',
            'Metal Gear Solid 4: Guns of the Patriots',
            'Destiny 2',
            'Game Boy vs. Gameboy',
            "Assassin's Creed",
            'Pokemon Sun and Moon',
            'PlayStation 3']
detail = []
for p in problems:
    row = {'text': p}
    for name, fn in vgir.TOKENIZERS.items():
        if name == 'whitespace': continue
        row[name] = ' | '.join(fn(p))
    detail.append(row)
vgir.write_csv(detail, 'tokenization_problems.csv')

import pandas as pd
print(pd.DataFrame(rows)); print()
print(pd.DataFrame(detail).to_string(index=False))

## §5 BPE — Ex. 4
Domain owner: B1. Runs here so the CSV is produced once; B1 copies `bpe_results.csv` from `10_PIPELINE_A/results_A/` to `20_PIPELINE_B/results_B/`.

GPT-2 pretrained BPE (via tiktoken, as in the BPE lab) vs BPE trained on our corpus (algorithm as in the lecture slides).

In [ ]:
from collections import Counter
corpus_words = Counter()
for d in docs.values():
    for w in vgir.tok_hybrid(vgir.full_text(d)):
        if '_' in w or w.isalpha(): corpus_words[w.lower()] += 1
merges = vgir.train_bpe(corpus_words, num_merges=1500)
print('trained merges:', len(merges))

probe = ['wolfenstein','playstation','metroid','nintendo','gameboy','counter-strike',
         'the','game','developed','1989','pokemon']
rows = []
for w in probe:
    try: gpt2 = vgir.bpe_gpt2(w)
    except RuntimeError as e: gpt2 = [f'[error: {e}]']
    trained = vgir.bpe_encode(w, merges)
    rows.append({'word': w, 'gpt2': ' | '.join(gpt2), 'gpt2_len': len(gpt2),
                 'corpus_bpe': ' | '.join(trained), 'corpus_len': len(trained)})
vgir.write_csv(rows, 'bpe_results.csv')
import pandas as pd; pd.DataFrame(rows)

## §6 Stemmers and lemmatizers with Paice OI/UI — Ex. 8, 10
Domain owner: B1. `OI` = over-stemming index, `UI` = under-stemming index (Paice 1994). Groups include one deliberate over-stemming trap: {universe, university, universities}, which Porter conflates.

In [ ]:
vgir.ensure_nltk()
from nltk.stem import PorterStemmer, SnowballStemmer, LancasterStemmer, WordNetLemmatizer

stemmers = {'porter': PorterStemmer().stem,
            'snowball': SnowballStemmer('english').stem,
            'lancaster': LancasterStemmer().stem}
wnl = WordNetLemmatizer()
groups = [
    ['develop','developer','developers','developing','development','developed'],
    ['release','releases','released','releasing'],
    ['play','plays','played','playing','player','players'],
    ['game','games','gaming','gamer','gamers'],
    ['universe','university','universities'],
]

def paice(fn, gs):
    stems = {w: fn(w) for g in gs for w in g}
    O = U = Om = Um = 0
    for i, gi in enumerate(gs):
        for a in range(len(gi)):
            for b in range(a + 1, len(gi)):
                Um += 1; U += stems[gi[a]] != stems[gi[b]]
        for j in range(i + 1, len(gs)):
            for a in gi:
                for b in gs[j]:
                    Om += 1; O += stems[a] == stems[b]
    UI = U / Um if Um else 0; OI = O / Om if Om else 0
    return {'UI': round(UI, 3), 'OI': round(OI, 3),
            'SW': round(OI / UI, 3) if UI else 0.0}

rows = [{'method': n, **paice(f, groups)} for n, f in stemmers.items()]
rows.append({'method': 'wordnet_v', **paice(lambda w: wnl.lemmatize(w, 'v'), groups)})
rows.append({'method': 'wordnet_n', **paice(lambda w: wnl.lemmatize(w, 'n'), groups)})
df = vgir.write_csv(rows, 'stemming_lemmatization.csv'); df

## §7 Order effect — Ex. 9
Domain owner: B1. Demonstrates that stemming before stop-word removal lets stems of stop words (e.g. Porter reduces `was` to `wa`) slip past the stop list.

In [ ]:
porter = PorterStemmer(); stops = vgir.stop_nltk()
sample = 'This was released and it has been played by many'.split()

stem_first = [porter.stem(w) for w in sample]
stem_first_stopped = [w for w in stem_first if w not in stops]

stop_first = [w for w in sample if w.lower() not in stops]
stop_first_stemmed = [porter.stem(w) for w in stop_first]

leaks = [w for w in stem_first_stopped if w in {'wa', 'thi', 'ha'}]
vgir.write_csv([
    {'order': 'stem_then_stop', 'output': ' '.join(stem_first_stopped),
     'kept_stopword_stems': ' '.join(leaks)},
    {'order': 'stop_then_stem', 'output': ' '.join(stop_first_stemmed),
     'kept_stopword_stems': ''}
], 'order_effect_stem_stop.csv')

print('input           :', sample)
print('stem then stop  :', stem_first_stopped, ' <- stems of stop words like wa/thi/ha slip past')
print('stop then stem  :', stop_first_stemmed)

## §8 (A2) POS tagging — Ex. 12
Requires `04_GOLD/pos_gold.tsv`. Format: one sentence per line, tokens as `word|TAG` (Penn Treebank). ~200 tokens total, including ≥ 1 occurrence each of Doom, Halo, Destiny, Prey as NNP (see PREREGISTRATION.md §2).

In [ ]:
POS_GOLD = f'{GOLD}/pos_gold.tsv'
if not os.path.exists(POS_GOLD):
    print('MISSING', POS_GOLD, '- A2 must create this before running §8. Skipping.')
else:
    gold = []
    with open(POS_GOLD) as f:
        for line in f:
            for wt in line.split():
                if '|' in wt: gold.append(tuple(wt.rsplit('|', 1)))
    print('gold tokens:', len(gold))
    if len(gold) < 100:
        print('WARNING: < 100 tokens - KNN cross-validation may fail. Extend the gold file.')
    words = [w for w, _ in gold]; y = [t for _, t in gold]

    import nltk
    nltk_pred = [t for _, t in nltk.pos_tag(words)]

    from spacy.tokens import Doc
    sp = vgir.spacy_model()
    doc = Doc(sp.vocab, words=words)
    for name, proc in sp.pipeline:
        if name in ('tok2vec', 'tagger', 'attribute_ruler'): doc = proc(doc)
    spacy_pred = [t.tag_ for t in doc]

    lx = set(w.lower() for w in vgir.lexicon_entries())
    def rule_tag(w):
        if w.lower() in lx: return 'NNP'
        if w.endswith('ing'): return 'VBG'
        if w.endswith('ed'):  return 'VBD'
        if w.endswith('ly'):  return 'RB'
        if w.endswith('s') and w[:-1].isalpha(): return 'NNS'
        if w[0:1].isupper(): return 'NNP'
        return 'NN'
    rule_pred = [rule_tag(w) for w in words]

    from sklearn.pipeline import Pipeline as SkP
    from sklearn.preprocessing import OneHotEncoder
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.model_selection import cross_val_predict
    feats = [[w, words[i-1] if i > 0 else '<S>',
              words[i+1] if i < len(words) - 1 else '</S>',
              str(len(w)), w[0], w[-1], str(w[:1].isupper())]
             for i, w in enumerate(words)]
    clf = SkP([('oh', OneHotEncoder(handle_unknown='ignore')),
               ('knn', KNeighborsClassifier(n_neighbors=3))])
    cv = min(5, max(2, len(gold) // 20))
    knn_pred = cross_val_predict(clf, feats, y, cv=cv).tolist()

    def acc(pred): return round(sum(p == g for p, (_, g) in zip(pred, gold)) / len(gold), 3)
    tagger_rows = [{'tagger': n, 'accuracy': acc(p)}
        for n, p in [('nltk_default', nltk_pred), ('spacy_default', spacy_pred),
                     ('rule_custom', rule_pred), ('knn_custom', knn_pred)]]
    vgir.write_csv(tagger_rows, 'pos_tagging_results.csv')
    import pandas as pd; print(pd.DataFrame(tagger_rows))

## §9 (B2) NER — Ex. 13
Requires `04_GOLD/ner_gold.tsv`. Header row `doc_id\ttext\tlabel`, 7 labels: PERSON, ORG, GPE, DATE, MONEY, PRODUCT, EVENT. spaCy sometimes labels games as WORK_OF_ART; the code maps that to PRODUCT.

In [ ]:
NER_GOLD = f'{GOLD}/ner_gold.tsv'
if not os.path.exists(NER_GOLD):
    print('MISSING', NER_GOLD, '- B2 must create this before running §9. Skipping.')
else:
    gold = []
    with open(NER_GOLD) as f:
        next(f, None)   # header
        for line in f:
            parts = line.rstrip('\n').split('\t')
            if len(parts) == 3: gold.append(tuple(parts))
    sp = vgir.spacy_model()
    by_doc = {}
    for did, txt, lab in gold: by_doc.setdefault(did, []).append((txt, lab))
    rows = []; C = I = M = 0
    for did, d in docs.items():
        pred = [(e.text, 'PRODUCT' if e.label_ == 'WORK_OF_ART' else e.label_)
                for e in sp(vgir.full_text(d)).ents]
        gset = set(by_doc.get(did, [])); pset = set(pred)
        C += len(gset & pset); I += len(pset - gset); M += len(gset - pset)
        for txt, lab in by_doc.get(did, []):
            if (txt, lab) in pset: status = 'correct'
            elif any(t == txt for t, _ in pred): status = 'mislabeled'
            else: status = 'missed'
            rows.append({'doc_id': did, 'entity': txt, 'gold_label': lab, 'status': status})
    P_ = C / (C + I) if (C + I) else 0.0
    R_ = C / len(gold) if gold else 0.0
    F_ = 2 * P_ * R_ / (P_ + R_) if (P_ + R_) else 0.0
    print(f'NER  P={P_:.3f}  R={R_:.3f}  F1={F_:.3f}  '
          f'(correct={C}, incorrect={I}, missed={M})')
    vgir.write_csv(rows, 'ner_results.csv')
    import pandas as pd; pd.DataFrame(rows).head(20)

## §10 (A2 for A, B2 for B) — n-grams 1 through 5
Runs both pipelines by default. This copy of the notebook keeps both so A2 can inspect the difference against Pipeline B.

In [ ]:
stops = vgir.stop_nltk()
sizes = {1: 'unigram', 2: 'bigram', 3: 'trigram', 4: 'quadgram', 5: 'quintgram'}
for pname, pfn in vgir.PIPELINES.items():
    ix = vgir.PositionalIndex(pfn, pname).build(docs)
    doc_terms = [ix.doc_terms[d] for d in docs]
    for n in (1, 2, 3, 4, 5):
        s = vgir.ngram_stats(doc_terms, n, top=10)
        rate = vgir.meaningful_rate(s['counter'], stops, top=20) if n >= 2 else None
        vgir.write_csv([{'gram': ' '.join(g), 'count': c} for g, c in s['top']],
                       f'{sizes[n]}_pipeline{pname}.csv')
        rr = f'  meaningful@top20={rate:.3f}' if rate is not None else ''
        print(f'Pipeline {pname} n={n}  total={s["total"]}  unique={s["unique"]}{rr}')

## §11 (A1) Before / after preprocessing — Table A row

In [ ]:
raw_tokens = sum(len(vgir.tok_hybrid(vgir.full_text(d))) for d in docs.values())
raw_unique = len({t for d in docs.values() for t in vgir.tok_hybrid(vgir.full_text(d))})
rows = [{'stage': 'raw (hybrid tokens, cased)', 'tokens': raw_tokens, 'vocabulary': raw_unique}]
for name, pfn in vgir.PIPELINES.items():
    all_terms = [t for d in docs.values() for t, _ in pfn(vgir.full_text(d))]
    rows.append({'stage': f'Pipeline {name}', 'tokens': len(all_terms),
                 'vocabulary': len(set(all_terms))})
df = vgir.write_csv(rows, 'preprocessing_results_A.csv'); df

---
## Outputs written to `10_PIPELINE_A/results_A/`

- `document_statistics.csv`, `preprocessing_results_A.csv`
- `tokenization_comparison.csv`, `tokenization_problems.csv`
- `bpe_results.csv` (B1 copies to `results_B/`)
- `stemming_lemmatization.csv`, `order_effect_stem_stop.csv` (B1 copies)
- `pos_tagging_results.csv` (once A2's `pos_gold.tsv` is present)
- `ner_results.csv` (once B2's `ner_gold.tsv` is present; B2 copies to `results_B/`)
- `{uni,bi,tri,quad,quint}gram_pipelineA.csv`, `{...}_pipelineB.csv` (B2 copies B files)

Hand off `03_CORPUS/` and `results_A/` to Notebook 2 (retrieval + evaluation + GUI).